# 🌿 AgroDetect — Entrenamiento del modelo de visión artificial

**Sistema Inteligente para la Detección de Enfermedades y Plagas Foliares en el Cultivo de Café**

UNAH Campus Comayagua (UNAH-CURC) — Asignatura: Inteligencia Artificial

Integrantes: Jeimy Jazmín Palma Santos · Ángeles Izamar Euceda Herrera · Kilver Said Nolasco Parada

Repositorio de datos: [github.com/Izamar-0302/AgroDetect](https://github.com/Izamar-0302/AgroDetect)

---

Este notebook cubre **todo el pipeline de visión artificial** descrito en el anteproyecto:

1. Descarga de los datasets desde el repositorio de GitHub del equipo
2. Extracción, organización y limpieza de las imágenes por clase
3. Análisis exploratorio del dataset
4. División en conjuntos de **entrenamiento / validación / prueba**
5. Preprocesamiento y **data augmentation**
6. Construcción del modelo con **Transfer Learning (MobileNetV2)**
7. Entrenamiento (fase congelada + fase de *fine-tuning*)
8. Evaluación: exactitud, precisión, sensibilidad, F1-score, matriz de confusión
9. Análisis de **umbrales de decisión** (confianza mínima para aceptar un diagnóstico)
10. Prueba del modelo con imágenes nuevas
11. Exportación del modelo entrenado para su uso en la app de Streamlit

> 💡 **Recomendado:** ejecutar en Google Colab con GPU activada
> (`Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`).


## 1. Configuración del entorno

Instalamos/confirmamos las librerías necesarias. TensorFlow, numpy, matplotlib y scikit-learn
ya vienen preinstalados en Colab; solo instalamos lo que falte.


In [ ]:
# Librerías del proyecto (ver Tabla 2 del anteproyecto: TensorFlow/Keras, OpenCV, Pillow,
# scikit-learn, pandas, matplotlib)
!pip install -q opencv-python-headless seaborn split-folders

import os
import sys
import shutil
import zipfile
import random
import json
import glob
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_recall_fscore_support, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))


TensorFlow: 2.20.0
GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Semillas para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)


## 2. Descarga de los datasets desde GitHub

Los datos están alojados en el repositorio del equipo:
[`github.com/Izamar-0302/AgroDetect`](https://github.com/Izamar-0302/AgroDetect)

El repositorio contiene, para cada clase, un archivo `.zip` con las imágenes:

| Archivo en el repo | Clase | Carpeta |
|---|---|---|
| `Dataset/coffee___healthy.zip.001/.002/.003` (dividido en 3 partes) | Hoja sana | `Dataset/` |
| `Dataset/coffee___rust.zip` | Roya (versión original) | `Dataset/` |
| `coffee___rust2.zip` | Roya (versión 2, en pruebas) | raíz del repo |
| `Dataset/coffee__phoma.zip` | Phoma | `Dataset/` |
| `Dataset/coffee__leaf_miner.zip` | Minador de la hoja | `Dataset/` |
| `cercospora.zip` | Cercospora | raíz del repo |
| `red_spider_mite.zip` | Araña roja | raíz del repo (almacenado con **Git LFS**) |

> 🔀 Para roya hay **dos fuentes disponibles**: la original (`Dataset/coffee___rust.zip`) y una
> segunda versión (`coffee___rust2.zip`) agregada para intentar mejorar la calidad/diversidad
> de esa clase. En la sección 3.2 se puede alternar entre ambas con la variable `USAR_RUST_V2`.

Como `red_spider_mite.zip` se subió mediante **Git LFS** (`.gitattributes` marca `*.zip` con `filter=lfs`),
la forma más confiable de traer *todos* los archivos reales (y no punteros LFS) es clonar el
repositorio con `git lfs` instalado, en lugar de descargar archivos sueltos.


In [2]:
# Instalamos git-lfs (necesario para resolver los archivos .zip grandes, en especial
# red_spider_mite.zip que se subió con Git LFS) y clonamos el repositorio del equipo.
!git lfs install
!apt-get -qq install git-lfs -y > /dev/null 2>&1 || true

REPO_URL = "https://github.com/Izamar-0302/AgroDetect.git"
REPO_DIR = "/content/AgroDetect"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone --depth 1 {REPO_URL} {REPO_DIR}
!cd {REPO_DIR} && git lfs pull

print("\nContenido descargado:")
for root, dirs, files in os.walk(REPO_DIR):
    dirs[:] = [d for d in dirs if d != '.git']
    for f in files:
        fp = os.path.join(root, f)
        if f.endswith('.zip') or '.zip.' in f:
            size_mb = os.path.getsize(fp) / (1024 * 1024)
            print(f"  {fp}  ({size_mb:.1f} MB)")


Git LFS initialized.


NameError: name 'os' is not defined

In [ ]:
# Si algún .zip quedó como puntero de Git LFS sin resolver (134 bytes aprox.),
# lo intentamos resolver de nuevo explícitamente.
def es_puntero_lfs(filepath):
    try:
        if os.path.getsize(filepath) > 1024:
            return False
        with open(filepath, 'rb') as f:
            inicio = f.read(200)
        return b'git-lfs' in inicio
    except Exception:
        return False

zips_repo = glob.glob(os.path.join(REPO_DIR, '**', '*.zip*'), recursive=True)
pendientes = [z for z in zips_repo if es_puntero_lfs(z)]

if pendientes:
    print("Archivos que aún son punteros LFS, forzando descarga:", pendientes)
    !cd {REPO_DIR} && git lfs pull --include="*"
else:
    print("✅ Todos los archivos .zip se descargaron con su contenido real.")


## 3. Reconstrucción de archivos divididos y extracción

El archivo de la clase **sana** (`coffee___healthy`) fue subido en 3 partes
(`.zip.001`, `.zip.002`, `.zip.003`) por su tamaño. Antes de extraerlo hay que
**concatenar los 3 fragmentos** en un único `.zip` válido.

Luego extraemos cada `.zip` y organizamos **todas** las imágenes en una única estructura
de carpetas por clase, sin importar cómo esté organizada internamente cada descarga
(algunas traen una subcarpeta, otras las imágenes sueltas). Esto hace el proceso robusto
ante diferencias de estructura entre los distintos repositorios de origen (Mendeley, Roboflow,
Kaggle, GitHub) mencionados en la sección 4.2 del anteproyecto.


In [ ]:
# Carpeta de trabajo para datos organizados
DATA_RAW_DIR = "/content/raw_zips"
DATA_ORG_DIR = "/content/dataset_organizado"

os.makedirs(DATA_RAW_DIR, exist_ok=True)
if os.path.exists(DATA_ORG_DIR):
    shutil.rmtree(DATA_ORG_DIR)
os.makedirs(DATA_ORG_DIR, exist_ok=True)

# Nombres de clase EN ESPAÑOL, alineados con la Tabla 1 del anteproyecto
CLASES = ["sana", "roya", "cercospora", "phoma", "arana_roja", "minador"]
for c in CLASES:
    os.makedirs(os.path.join(DATA_ORG_DIR, c), exist_ok=True)

print("Clases del proyecto:", CLASES)


In [ ]:
# 3.1 Reconstruimos coffee___healthy.zip a partir de sus 3 partes
healthy_parts = sorted(glob.glob(os.path.join(REPO_DIR, "Dataset", "coffee___healthy.zip.*")))
assert len(healthy_parts) == 3, f"Se esperaban 3 partes, se encontraron {len(healthy_parts)}: {healthy_parts}"

healthy_zip_path = os.path.join(DATA_RAW_DIR, "coffee___healthy.zip")
with open(healthy_zip_path, "wb") as out:
    for part in healthy_parts:
        with open(part, "rb") as p:
            out.write(p.read())

print("coffee___healthy.zip reconstruido:", os.path.getsize(healthy_zip_path) / (1024*1024), "MB")

# Verificamos que sea un zip válido
assert zipfile.is_zipfile(healthy_zip_path), "La reconstrucción del zip de 'sana' falló"
print("✅ Zip reconstruido correctamente")


In [ ]:
# 3.2 Mapa: clase -> ruta del archivo .zip de origen
#
# Selector para roya: coffee___rust2.zip es una segunda fuente/version del dataset de roya
# que se subio fuera de la carpeta Dataset/ (en la raiz del repo, junto a cercospora.zip y
# red_spider_mite.zip). Se deja como interruptor para poder alternar facilmente entre ambas
# versiones y comparar resultados sin tener que editar rutas a mano cada vez.
USAR_RUST_V2 = True  # True = usa coffee___rust2.zip (nueva version, raiz del repo)
                      # False = usa el coffee___rust.zip original (dentro de Dataset/)

ruta_roya = (
    os.path.join(REPO_DIR, "coffee___rust2.zip") if USAR_RUST_V2
    else os.path.join(REPO_DIR, "Dataset", "coffee___rust.zip")
)

ZIP_SOURCES = {
    "sana":       healthy_zip_path,
    "roya":       ruta_roya,
    "phoma":      os.path.join(REPO_DIR, "Dataset", "coffee__phoma.zip"),
    "minador":    os.path.join(REPO_DIR, "Dataset", "coffee__leaf_miner.zip"),
    "cercospora": os.path.join(REPO_DIR, "cercospora.zip"),
    "arana_roja": os.path.join(REPO_DIR, "Dataset",  "red_spider_mite.zip"),
}

print(f"🔀 Fuente de roya seleccionada: {'coffee___rust2.zip (nueva)' if USAR_RUST_V2 else 'coffee___rust.zip (original)'}\n")

for clase, ruta in ZIP_SOURCES.items():
    existe = os.path.exists(ruta)
    tam = os.path.getsize(ruta) / (1024*1024) if existe else 0
    print(f"{clase:12s} -> {ruta}  (existe={existe}, {tam:.1f} MB)")


In [ ]:
# 3.3 Extracción robusta: para cada zip, buscamos TODAS las imágenes de forma
# recursiva (sin importar la subcarpeta interna) y las copiamos, ya renombradas,
# a dataset_organizado/<clase>/

IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp')

def extraer_y_organizar(clase, zip_path, destino_dir):
    tmp_dir = f"/content/_tmp_extract_{clase}"
    if os.path.exists(tmp_dir):
        shutil.rmtree(tmp_dir)
    os.makedirs(tmp_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(tmp_dir)

    contador = 0
    for root, dirs, files in os.walk(tmp_dir):
        if '__MACOSX' in root:
            continue
        for fname in files:
            if fname.lower().endswith(IMG_EXT) and not fname.startswith('.'):
                origen = os.path.join(root, fname)
                ext = os.path.splitext(fname)[1].lower()
                destino = os.path.join(destino_dir, f"{clase}_{contador:05d}{ext}")
                shutil.copy2(origen, destino)
                contador += 1

    shutil.rmtree(tmp_dir)
    return contador

resumen_extraccion = {}
for clase, zip_path in ZIP_SOURCES.items():
    destino = os.path.join(DATA_ORG_DIR, clase)
    n = extraer_y_organizar(clase, zip_path, destino)
    resumen_extraccion[clase] = n
    print(f"✅ {clase:12s}: {n} imágenes extraídas")


In [ ]:
# 3.4 Filtramos imágenes corruptas o ilegibles antes de continuar
def imagen_valida(path):
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False

eliminadas = 0
for clase in CLASES:
    carpeta = os.path.join(DATA_ORG_DIR, clase)
    for fname in os.listdir(carpeta):
        fpath = os.path.join(carpeta, fname)
        if not imagen_valida(fpath):
            os.remove(fpath)
            eliminadas += 1

print(f"Imágenes corruptas eliminadas: {eliminadas}")


## 4. Análisis exploratorio del dataset

Revisamos cuántas imágenes quedaron por clase (para detectar desbalance) y visualizamos
ejemplos de cada categoría.


In [ ]:
conteo_final = {c: len(os.listdir(os.path.join(DATA_ORG_DIR, c))) for c in CLASES}
df_conteo = pd.DataFrame(list(conteo_final.items()), columns=["clase", "num_imagenes"])
df_conteo = df_conteo.sort_values("num_imagenes", ascending=False).reset_index(drop=True)
print(df_conteo)
print("\nTotal de imágenes en el dataset:", df_conteo["num_imagenes"].sum())


In [ ]:
plt.figure(figsize=(9, 5))
colores = sns.color_palette("YlGn", len(df_conteo))
sns.barplot(data=df_conteo, x="clase", y="num_imagenes", palette=colores)
plt.title("Cantidad de imágenes por clase — AgroDetect")
plt.xlabel("Clase")
plt.ylabel("Número de imágenes")
plt.xticks(rotation=20)
for i, v in enumerate(df_conteo["num_imagenes"]):
    plt.text(i, v + 3, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

# Nota: si alguna clase tiene muchas menos imágenes que las demás, el desbalance
# se compensará más adelante con pesos de clase (class_weight) durante el entrenamiento.


In [ ]:
# Mostramos ejemplos aleatorios de cada clase
fig, axes = plt.subplots(len(CLASES), 4, figsize=(14, 3.2 * len(CLASES)))
for i, clase in enumerate(CLASES):
    carpeta = os.path.join(DATA_ORG_DIR, clase)
    archivos = os.listdir(carpeta)
    muestra = random.sample(archivos, min(4, len(archivos)))
    for j, fname in enumerate(muestra):
        img = Image.open(os.path.join(carpeta, fname))
        axes[i, j].imshow(img)
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_ylabel(clase, fontsize=12)
        axes[i, j].set_title(clase if j == 0 else "", fontsize=10)
plt.tight_layout()
plt.show()


## 5. División en Train / Validation / Test

Dividimos el dataset de forma **estratificada** (misma proporción de clases en cada
conjunto) siguiendo un esquema **70% entrenamiento / 15% validación / 15% prueba**,
tal como se describe en la fase de *Evaluación* del anteproyecto (sección 4.1, paso 5).


In [ ]:
SPLIT_DIR = "/content/dataset_split"
if os.path.exists(SPLIT_DIR):
    shutil.rmtree(SPLIT_DIR)

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

for subset in ["train", "val", "test"]:
    for clase in CLASES:
        os.makedirs(os.path.join(SPLIT_DIR, subset, clase), exist_ok=True)

resumen_split = []
for clase in CLASES:
    carpeta = os.path.join(DATA_ORG_DIR, clase)
    archivos = os.listdir(carpeta)

    # 70% train, 30% restante -> luego partido en val/test (15%/15%)
    train_files, resto_files = train_test_split(
        archivos, train_size=TRAIN_RATIO, random_state=SEED
    )
    val_files, test_files = train_test_split(
        resto_files, train_size=VAL_RATIO / (VAL_RATIO + TEST_RATIO), random_state=SEED
    )

    for subset, files in [("train", train_files), ("val", val_files), ("test", test_files)]:
        for fname in files:
            shutil.copy2(
                os.path.join(carpeta, fname),
                os.path.join(SPLIT_DIR, subset, clase, fname)
            )

    resumen_split.append({
        "clase": clase, "train": len(train_files),
        "val": len(val_files), "test": len(test_files),
        "total": len(archivos)
    })

df_split = pd.DataFrame(resumen_split)
print(df_split)
print("\nTotales -> train:", df_split["train"].sum(),
      " val:", df_split["val"].sum(),
      " test:", df_split["test"].sum())


### 5.1 Balanceo del conjunto de entrenamiento (generación de imágenes sintéticas)

La meta original del anteproyecto era contar con **~400 imágenes por clase**. Al revisar el
split real, algunas clases ya alcanzan o superan esa meta (p. ej. phoma, minador), pero otras
quedan por debajo — notablemente **cercospora** y sobre todo **araña_roja**, que es también la
clase con peor F1-score en la evaluación.

Para acercar todas las clases a la meta de 400, generamos imágenes sintéticas mediante
transformaciones aleatorias (volteo, rotación, zoom/recorte, brillo, contraste y saturación)
a partir de las imágenes reales **ya existentes en train**, hasta alcanzar la proporción
correspondiente de la meta (400 × 70% ≈ 280 imágenes por clase en train).

⚠️ **Importante — por qué solo se aplica a `train`:** si generáramos las imágenes sintéticas
*antes* de dividir en train/val/test, podría quedar una imagen real en el conjunto de prueba y
una copia sintética casi idéntica en el de entrenamiento (*data leakage*), lo que infla
artificialmente las métricas de validación/prueba. Por eso este balanceo se hace **después**
del split de la sección 5, y **solo sobre la carpeta de `train`**; `val` y `test` se mantienen
100% con imágenes reales para que la evaluación siga siendo honesta.

⚠️ **Lo que esto NO resuelve:** roya ya tenía ~400 imágenes en el dataset original y aun así
tuvo un F1 bajo (0.65); su confusión con sana y araña_roja en la matriz de confusión sugiere un
problema de **solapamiento visual entre clases**, no de cantidad de datos. Generar más copias
sintéticas de roya no va a arreglar eso — para mejorarlo haría falta revisar la calidad/
diversidad real de esas imágenes, o incluso replantear si alguna subclase de roya está mal
etiquetada.


In [ ]:
import random
from PIL import ImageEnhance

TARGET_TOTAL_POR_CLASE = 400  # meta original del anteproyecto (imagenes reales + sinteticas)
TARGET_TRAIN_POR_CLASE = round(TARGET_TOTAL_POR_CLASE * TRAIN_RATIO)  # ~280 en train

print(f"Meta de imagenes en TRAIN por clase: {TARGET_TRAIN_POR_CLASE}")
print("(val y test NO se tocan: se mantienen 100% imagenes reales para una evaluacion honesta)\n")

def generar_imagen_aumentada(img):
    """Aplica una combinacion aleatoria de transformaciones a una imagen PIL (RGB)."""
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    angulo = random.uniform(-25, 25)
    img = img.rotate(angulo, expand=False, fillcolor=(255, 255, 255))

    ancho, alto = img.size
    zoom = random.uniform(0.85, 1.0)
    nx, ny = max(1, int(ancho * zoom)), max(1, int(alto * zoom))
    x0 = random.randint(0, ancho - nx)
    y0 = random.randint(0, alto - ny)
    img = img.crop((x0, y0, x0 + nx, y0 + ny)).resize((ancho, alto))

    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.2))
    img = ImageEnhance.Color(img).enhance(random.uniform(0.85, 1.15))
    return img

resumen_balanceo = []
for clase in CLASES:
    carpeta_train = os.path.join(SPLIT_DIR, "train", clase)
    archivos_reales = [f for f in os.listdir(carpeta_train)]
    n_actual = len(archivos_reales)
    n_faltantes = max(0, TARGET_TRAIN_POR_CLASE - n_actual)

    generadas = 0
    if n_faltantes > 0 and archivos_reales:
        for i in range(n_faltantes):
            origen_fname = random.choice(archivos_reales)
            with Image.open(os.path.join(carpeta_train, origen_fname)).convert("RGB") as img:
                img_aum = generar_imagen_aumentada(img)
            nuevo_nombre = f"{clase}_sint_{i:05d}.jpg"
            img_aum.save(os.path.join(carpeta_train, nuevo_nombre), quality=90)
            generadas += 1

    resumen_balanceo.append({
        "clase": clase, "reales_train": n_actual,
        "sinteticas_generadas": generadas,
        "total_train_final": n_actual + generadas,
    })
    print(f"{clase:12s}: {n_actual:4d} reales + {generadas:4d} sinteticas = {n_actual + generadas:4d} en train")

df_balanceo = pd.DataFrame(resumen_balanceo)
df_balanceo


In [ ]:
# Verificacion visual rapida: comparamos una imagen real vs. una sintetica de la
# clase con mas imagenes generadas, para confirmar que el augmentation se ve razonable.
clase_revisar = df_balanceo.sort_values("sinteticas_generadas", ascending=False).iloc[0]["clase"]
carpeta = os.path.join(SPLIT_DIR, "train", clase_revisar)
reales = [f for f in os.listdir(carpeta) if "_sint_" not in f]
sinteticas = [f for f in os.listdir(carpeta) if "_sint_" in f]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, fname in zip(axes[:2], random.sample(reales, min(2, len(reales)))):
    ax.imshow(Image.open(os.path.join(carpeta, fname)))
    ax.set_title("Real")
    ax.axis("off")
for ax, fname in zip(axes[2:], random.sample(sinteticas, min(2, len(sinteticas)))):
    ax.imshow(Image.open(os.path.join(carpeta, fname)))
    ax.set_title("Sintética")
    ax.axis("off")
plt.suptitle(f"Clase: {clase_revisar}")
plt.tight_layout()
plt.show()


## 6. Preprocesamiento y *data augmentation*

Cargamos las imágenes con `image_dataset_from_directory` y aplicamos **aumento de datos**
(rotaciones, volteos horizontales y ajustes de brillo/contraste) al conjunto de entrenamiento,
como se especifica en la sección 4.1 del anteproyecto, para mejorar la robustez del modelo ante
distintas condiciones de campo (ángulo de la foto, iluminación, etc.).

La normalización final de los píxeles **no se aplica todavía aquí**: como en la sección 7 se
comparan tres arquitecturas distintas (MobileNetV2, EfficientNetB0 y ResNet50), y cada una
espera su propio rango de entrada, el `preprocess_input` correspondiente se aplica más adelante,
de forma independiente para cada modelo.


In [ ]:
IMG_SIZE = (224, 224)   # tamaño de entrada estándar de MobileNetV2
BATCH_SIZE = 32

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(SPLIT_DIR, "train"),
    labels="inferred",
    label_mode="categorical",
    class_names=CLASES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(SPLIT_DIR, "val"),
    labels="inferred",
    label_mode="categorical",
    class_names=CLASES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(SPLIT_DIR, "test"),
    labels="inferred",
    label_mode="categorical",
    class_names=CLASES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

NUM_CLASES = len(CLASES)
print("Clases (orden usado por el modelo):", train_ds_raw.class_names)


In [ ]:
# Capa de aumento de datos: rotaciones, volteos y ajustes de brillo/contraste
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),          # rotaciones
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),          # ajuste de contraste
    layers.RandomBrightness(0.15),        # ajuste de brillo
], name="data_augmentation")

def aplicar_augmentation(ds):
    """Aplica solo el aumento de datos (imagenes aun en escala 0-255)."""
    return ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                  num_parallel_calls=tf.data.AUTOTUNE)

# Conjuntos "en crudo": entrenamiento aumentado, validacion/prueba sin tocar.
# El preprocess_input especifico de cada arquitectura se aplica despues (seccion 7),
# ya que MobileNetV2, EfficientNetB0 y ResNet50 normalizan los pixeles de forma distinta.
#
# Nota: NO usamos .cache() aqui a proposito. Con un dataset grande, cachear en RAM
# puede saturar la memoria de Colab (y ademas 'congelaria' el augmentation aleatorio,
# haciendo que cada epoca vea siempre las mismas imagenes aumentadas). Si tu dataset
# es pequeno y quieres acelerar, puedes agregar .cache() de vuelta.
train_ds_aug = aplicar_augmentation(train_ds_raw)
val_ds_cache = val_ds_raw
test_ds_cache = test_ds_raw

def construir_pipeline(preprocess_fn):
    """Construye (train_ds, val_ds, test_ds) aplicando el preprocess_input de una
    arquitectura especifica sobre los conjuntos ya divididos y aumentados."""
    tr = (train_ds_aug
          .map(lambda x, y: (preprocess_fn(x), y), num_parallel_calls=tf.data.AUTOTUNE)
          .prefetch(tf.data.AUTOTUNE))
    va = (val_ds_cache
          .map(lambda x, y: (preprocess_fn(x), y), num_parallel_calls=tf.data.AUTOTUNE)
          .prefetch(tf.data.AUTOTUNE))
    te = (test_ds_cache
          .map(lambda x, y: (preprocess_fn(x), y), num_parallel_calls=tf.data.AUTOTUNE)
          .prefetch(tf.data.AUTOTUNE))
    return tr, va, te


In [ ]:
# Visualizamos el efecto del data augmentation sobre una imagen de ejemplo
for images, labels in train_ds_raw.take(1):
    imagen_original = images[0].numpy().astype("uint8")
    break

plt.figure(figsize=(12, 4))
plt.subplot(1, 5, 1)
plt.imshow(imagen_original)
plt.title("Original")
plt.axis('off')

for i in range(4):
    aug_img = data_augmentation(tf.expand_dims(imagen_original, 0), training=True)
    plt.subplot(1, 5, i + 2)
    plt.imshow(aug_img[0].numpy().astype("uint8"))
    plt.title(f"Aumentada {i+1}")
    plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Pesos de clase, por si el dataset queda desbalanceado tras la extracción real
from sklearn.utils.class_weight import compute_class_weight

etiquetas_train = []
for clase_idx, clase in enumerate(CLASES):
    carpeta = os.path.join(SPLIT_DIR, "train", clase)
    etiquetas_train += [clase_idx] * len(os.listdir(carpeta))

pesos = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASES),
    y=etiquetas_train
)
class_weight = {i: w for i, w in enumerate(pesos)}
print("Pesos de clase (class_weight):")
for i, clase in enumerate(CLASES):
    print(f"  {clase:12s}: {class_weight[i]:.3f}")


## 7. Comparación de arquitecturas candidatas (Transfer Learning)

Siguiendo la sección 4.3 del anteproyecto, en lugar de fijar una sola arquitectura se entrenan y
comparan **tres redes preentrenadas en ImageNet** como extractores de características
(*transfer learning*), todas bajo la misma estrategia de dos fases:

- **Fase 1 (feature extraction):** se entrena solo la cabeza de clasificación, con la base
  congelada.
- **Fase 2 (fine-tuning):** se descongelan las últimas capas de la base y se reentrena todo el
  modelo con una tasa de aprendizaje mucho más baja.

Las arquitecturas evaluadas son:

| Arquitectura | Motivación |
|---|---|
| **MobileNetV2** | Liviana y rápida; ideal para despliegue en la nube gratuita (Streamlit Community Cloud) y para un eventual uso móvil. |
| **EfficientNetB0** | Buen equilibrio entre exactitud y número de parámetros; suele superar a MobileNetV2 en precisión con un costo moderado. |
| **ResNet50** | Arquitectura profunda de referencia en visión por computador; sirve como punto de comparación de mayor capacidad. |

Al final se selecciona automáticamente la arquitectura con mejor **F1-score sobre el conjunto de
prueba**, por ser una métrica balanceada entre precisión y sensibilidad, relevante dado que el
dataset combina varias fuentes y puede quedar desbalanceado entre clases.


In [ ]:
from tensorflow.keras.applications import EfficientNetB0, ResNet50
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
import time

# Definimos las 3 arquitecturas candidatas, su funcion de preprocesamiento propia y
# cuantas capas finales de la base se descongelaran en la fase de fine-tuning.
ARQUITECTURAS = {
    "MobileNetV2":    {"clase": MobileNetV2,    "preprocess": mobilenet_preprocess, "capas_descongelar": 40},
    "EfficientNetB0": {"clase": EfficientNetB0, "preprocess": effnet_preprocess,    "capas_descongelar": 30},
    "ResNet50":       {"clase": ResNet50,       "preprocess": resnet_preprocess,    "capas_descongelar": 15},
}


In [ ]:
def entrenar_arquitectura(nombre, config, epochs_fase1=5, epochs_fase2=3, pasos_por_epoca=None):
    """Construye, entrena (2 fases) y evalua una arquitectura candidata.
    Devuelve un diccionario con el modelo, su historial y sus metricas de prueba."""
    print(f"\n{'='*70}\nEntrenando arquitectura: {nombre}\n{'='*70}")

    tr_ds, va_ds, te_ds = construir_pipeline(config["preprocess"])

    base = config["clase"](input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
    base.trainable = False  # Fase 1: extractor de caracteristicas congelado

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(NUM_CLASES, activation="softmax")(x)
    modelo = keras.Model(inputs, outputs, name=f"AgroDetect_{nombre}")

    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")]
    )

    cb_fase1 = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6),
    ]

    inicio = time.time()
    print(f"--- Fase 1: cabeza de clasificacion (base congelada) ---")
    hist1 = modelo.fit(tr_ds, validation_data=va_ds, epochs=epochs_fase1,
                        steps_per_epoch=pasos_por_epoca,
                        class_weight=class_weight, callbacks=cb_fase1, verbose=1)

    # === Fase 2: fine-tuning ===
    base.trainable = True
    fine_tune_at = len(base.layers) - config["capas_descongelar"]
    for layer in base.layers[:fine_tune_at]:
        layer.trainable = False

    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss="categorical_crossentropy",
        metrics=["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")]
    )

    cb_fase2 = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-7),
    ]

    print(f"--- Fase 2: fine-tuning de las ultimas {config['capas_descongelar']} capas ---")
    hist2 = modelo.fit(tr_ds, validation_data=va_ds, epochs=epochs_fase1 + epochs_fase2,
                        initial_epoch=hist1.epoch[-1] + 1,
                        steps_per_epoch=pasos_por_epoca,
                        class_weight=class_weight, callbacks=cb_fase2, verbose=1)
    tiempo_total = time.time() - inicio

    test_loss, test_acc, test_prec, test_rec = modelo.evaluate(te_ds, verbose=0)
    test_f1 = 2 * test_prec * test_rec / (test_prec + test_rec + 1e-9)

    print(f"\n{nombre} -> accuracy: {test_acc:.4f} | precision: {test_prec:.4f} | "
          f"recall: {test_rec:.4f} | F1: {test_f1:.4f} | tiempo: {tiempo_total/60:.1f} min")

    return {
        "nombre": nombre, "model": modelo,
        "history_fase1": hist1, "history_fase2": hist2,
        "train_ds": tr_ds, "val_ds": va_ds, "test_ds": te_ds,
        "preprocess_fn": config["preprocess"],
        "test_loss": test_loss, "test_acc": test_acc,
        "test_prec": test_prec, "test_rec": test_rec, "test_f1": test_f1,
        "num_parametros": modelo.count_params(), "tiempo_seg": tiempo_total,
    }


### Entrenamiento de las 3 arquitecturas

> ⏱️ Como el dataset es grande, ya se redujeron los valores por defecto
> (`epochs_fase1=5`, `epochs_fase2=3`, `patience` mas corta) respecto a la version anterior,
> y se quito el `.cache()` en RAM del pipeline para evitar quedarse sin memoria. Aun asi,
> entrenar 3 arquitecturas completas puede tardar. Si tu dataset es muy grande, activa
> `MODO_RAPIDO = True` en la celda siguiente para primero comparar las 3 arquitecturas
> usando solo una fraccion del set de entrenamiento (util para elegir la ganadora rapido);
> luego puedes reentrenar solo la arquitectura ganadora con el dataset completo y mas epocas
> en la seccion 8.


In [ ]:
# Si tu dataset es grande y quieres primero elegir la arquitectura ganadora rapido,
# activa MODO_RAPIDO (limita cuantos lotes/batches se usan por epoca durante la
# comparacion). Luego, una vez elegida la arquitectura, puedes reentrenarla con
# MODO_RAPIDO = False (dataset completo) y mas epocas para el modelo final.
MODO_RAPIDO = False
PASOS_RAPIDOS = 40  # numero de batches por epoca cuando MODO_RAPIDO = True

resultados = {}
for nombre, config in ARQUITECTURAS.items():
    resultados[nombre] = entrenar_arquitectura(
        nombre, config,
        pasos_por_epoca=PASOS_RAPIDOS if MODO_RAPIDO else None,
    )

print("\n✅ Entrenamiento de las 3 arquitecturas completado.")


### Tabla comparativa y selección de la mejor arquitectura

In [ ]:
tabla_comparativa = pd.DataFrame([
    {
        "Arquitectura": r["nombre"],
        "Exactitud (test)": r["test_acc"],
        "Precision (test)": r["test_prec"],
        "Sensibilidad (test)": r["test_rec"],
        "F1-score (test)": r["test_f1"],
        "Parametros (millones)": round(r["num_parametros"] / 1e6, 2),
        "Tiempo entrenamiento (min)": round(r["tiempo_seg"] / 60, 1),
    }
    for r in resultados.values()
]).sort_values("F1-score (test)", ascending=False).reset_index(drop=True)

print("📊 Comparación de arquitecturas evaluadas (ordenadas por F1-score):\n")
print(tabla_comparativa.to_string(index=False))

plt.figure(figsize=(9, 5))
colores_barras = ["#2e6b1f", "#4a7c3f", "#8fbf7a"]
plt.bar(tabla_comparativa["Arquitectura"], tabla_comparativa["F1-score (test)"], color=colores_barras)
plt.ylabel("F1-score (conjunto de prueba)")
plt.title("Comparación de arquitecturas candidatas")
plt.ylim(0, 1)
for i, v in enumerate(tabla_comparativa["F1-score (test)"]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Seleccionamos la arquitectura ganadora combinando desempeño (F1) y tamaño del
# modelo, ya que el destino final es desplegar en Streamlit Community Cloud (tier
# gratuito, RAM limitada). Un modelo con F1 ligeramente menor pero mucho mas liviano
# puede ser mejor decision practica que el F1 mas alto a secas.
PRIORIZAR_DESPLIEGUE = True  # False = elegir solo por F1-score, sin importar el tamaño

tabla_comparativa["F1_norm"] = (
    tabla_comparativa["F1-score (test)"] / tabla_comparativa["F1-score (test)"].max()
)
tabla_comparativa["Tamano_norm"] = (
    tabla_comparativa["Parametros (millones)"].min() / tabla_comparativa["Parametros (millones)"]
)
tabla_comparativa["Puntaje_despliegue"] = (
    0.7 * tabla_comparativa["F1_norm"] + 0.3 * tabla_comparativa["Tamano_norm"]
)

criterio_orden = "Puntaje_despliegue" if PRIORIZAR_DESPLIEGUE else "F1-score (test)"
tabla_comparativa = tabla_comparativa.sort_values(criterio_orden, ascending=False).reset_index(drop=True)

print("📊 Tabla comparativa (ordenada por", criterio_orden, "):\n")
print(tabla_comparativa[["Arquitectura", "F1-score (test)", "Parametros (millones)",
                          "Puntaje_despliegue"]].to_string(index=False))

# Seleccionamos la arquitectura ganadora y la dejamos asignada a las variables
# "model", "train_ds", "val_ds", "test_ds" y "preprocess_input" para que el resto
# del notebook (evaluacion, umbrales, pruebas y exportacion) funcione exactamente
# igual sin importar cual arquitectura haya resultado ganadora.
mejor_nombre = tabla_comparativa.iloc[0]["Arquitectura"]
print(f"\n🏆 Arquitectura seleccionada para el despliegue: {mejor_nombre}")

mejor = resultados[mejor_nombre]
model = mejor["model"]
history_fase1 = mejor["history_fase1"]
history_fase2 = mejor["history_fase2"]
train_ds = mejor["train_ds"]
val_ds = mejor["val_ds"]
test_ds = mejor["test_ds"]
preprocess_input = mejor["preprocess_fn"]  # usado mas adelante en diagnosticar()


## 8. Curvas de entrenamiento del modelo seleccionado

Graficamos la exactitud y la pérdida por época del modelo ganador (fases 1 y 2 combinadas),
para verificar que no hubo sobreajuste evidente antes de pasar a la evaluación formal.


In [ ]:
# Combinamos el historial de ambas fases del modelo ganador para graficar el entrenamiento completo
def combinar_historiales(h1, h2):
    combinado = {}
    for key in h1.history:
        combinado[key] = h1.history[key] + h2.history.get(key, [])
    return combinado

historial = combinar_historiales(history_fase1, history_fase2)
epoca_finetune = len(history_fase1.epoch)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(historial["accuracy"], label="Entrenamiento")
axes[0].plot(historial["val_accuracy"], label="Validación")
axes[0].axvline(epoca_finetune, color="gray", linestyle="--", label="Inicio fine-tuning")
axes[0].set_title(f"Exactitud (Accuracy) por época — {mejor_nombre}")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(historial["loss"], label="Entrenamiento")
axes[1].plot(historial["val_loss"], label="Validación")
axes[1].axvline(epoca_finetune, color="gray", linestyle="--", label="Inicio fine-tuning")
axes[1].set_title(f"Pérdida (Loss) por época — {mejor_nombre}")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()


## 9. Evaluación del modelo seleccionado

Evaluamos el modelo ganador (la arquitectura elegida como mejor en la sección 7) sobre el
**conjunto de prueba** (test), que nunca vio durante el entrenamiento, usando las métricas
estándar mencionadas en el objetivo específico del anteproyecto: **exactitud, precisión,
sensibilidad (recall) y F1-score**.


In [ ]:
test_loss, test_acc, test_prec, test_rec = model.evaluate(test_ds)
print(f"\n📊 Resultados sobre el conjunto de PRUEBA:")
print(f"  Exactitud (accuracy):  {test_acc:.4f}")
print(f"  Precisión (precision): {test_prec:.4f}")
print(f"  Sensibilidad (recall): {test_rec:.4f}")
print(f"  Pérdida (loss):        {test_loss:.4f}")


In [ ]:
# Predicciones sobre el conjunto de prueba (probabilidades por clase)
y_true = []
y_probs = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_probs.append(preds)
    y_true.append(labels.numpy())

y_probs = np.concatenate(y_probs, axis=0)
y_true = np.concatenate(y_true, axis=0)

y_true_idx = np.argmax(y_true, axis=1)
y_pred_idx = np.argmax(y_probs, axis=1)
confianza_pred = np.max(y_probs, axis=1)   # probabilidad de la clase predicha (para umbrales, sección 10)

print("Predicciones generadas para", len(y_true_idx), "imágenes de prueba")


In [ ]:
print("📋 Reporte de clasificación (test):\n")
print(classification_report(y_true_idx, y_pred_idx, target_names=CLASES, digits=3))


In [ ]:
cm = confusion_matrix(y_true_idx, y_pred_idx)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASES)
disp.plot(ax=ax, cmap="Greens", xticks_rotation=30, colorbar=True)
plt.title("Matriz de confusión — Conjunto de prueba")
plt.tight_layout()
plt.show()


### 9.1 Diagnóstico visual: revisión manual de errores en araña_roja

Araña_roja ha sido la clase con peor desempeño en las 4 corridas realizadas, sin importar la
arquitectura ni el balanceo del dataset. Antes de seguir ajustando el modelo, revisamos a ojo
las imágenes reales que el modelo confunde, para distinguir si el problema es del **modelo**
(no logra aprender el patrón) o del **dato** (imágenes ambiguas, mal recortadas, o síntomas muy
similares a otra clase en una etapa temprana).


In [ ]:
# El orden de test_ds_raw.file_paths coincide con y_true_idx / y_pred_idx porque
# test_ds_raw se creo con shuffle=False y nunca se reordena en el pipeline.
archivos_test = test_ds_raw.file_paths

idx_arana = CLASES.index("arana_roja")
es_arana_real = (y_true_idx == idx_arana)
mal_clasificadas = es_arana_real & (y_pred_idx != idx_arana)
bien_clasificadas = es_arana_real & (y_pred_idx == idx_arana)

indices_mal = np.where(mal_clasificadas)[0]
indices_bien = np.where(bien_clasificadas)[0]

print(f"araña_roja en test: {es_arana_real.sum()} imágenes")
print(f"  -> correctas:   {len(indices_bien)}")
print(f"  -> incorrectas: {len(indices_mal)}")


In [ ]:
# Mostramos las imagenes MAL clasificadas (araña_roja real, predicha como otra cosa)
n_mostrar = min(12, len(indices_mal))
cols = 4
filas = max(1, int(np.ceil(n_mostrar / cols)))
fig, axes = plt.subplots(filas, cols, figsize=(4 * cols, 4 * filas))
axes = np.array(axes).reshape(-1)

for ax, idx in zip(axes, indices_mal[:n_mostrar]):
    img = Image.open(archivos_test[idx])
    ax.imshow(img)
    ax.set_title(f"Real: araña_roja\nPred: {CLASES[y_pred_idx[idx]]} ({confianza_pred[idx]:.0%})", fontsize=9)
    ax.axis("off")
for ax in axes[n_mostrar:]:
    ax.axis("off")

plt.suptitle("araña_roja clasificada INCORRECTAMENTE (conjunto de prueba)", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Para comparar, mostramos tambien algunas BIEN clasificadas de araña_roja
n_mostrar_ok = min(8, len(indices_bien))
cols = 4
filas = max(1, int(np.ceil(n_mostrar_ok / cols)))
fig, axes = plt.subplots(filas, cols, figsize=(4 * cols, 4 * filas))
axes = np.array(axes).reshape(-1)

for ax, idx in zip(axes, indices_bien[:n_mostrar_ok]):
    img = Image.open(archivos_test[idx])
    ax.imshow(img)
    ax.set_title(f"Real: araña_roja\nPred: correcto ({confianza_pred[idx]:.0%})", fontsize=9)
    ax.axis("off")
for ax in axes[n_mostrar_ok:]:
    ax.axis("off")

plt.suptitle("araña_roja clasificada CORRECTAMENTE, para comparar", fontsize=13)
plt.tight_layout()
plt.show()

print("👀 Revisa: ¿las mal clasificadas se ven genuinamente distintas/ambiguas respecto a las")
print("   bien clasificadas? ¿hay fotos borrosas, mal recortadas, con varias hojas, u otro")
print("   problema de calidad? Eso orienta si conviene limpiar el dataset o revisar etiquetas.")


In [ ]:
# Precisión, sensibilidad y F1-score por clase, en tabla, para incluir en el informe
precision, recall, f1, soporte = precision_recall_fscore_support(y_true_idx, y_pred_idx, labels=range(NUM_CLASES))
df_metricas = pd.DataFrame({
    "clase": CLASES,
    "precision": precision,
    "recall (sensibilidad)": recall,
    "f1-score": f1,
    "soporte (n° imágenes)": soporte,
})
df_metricas.loc["promedio"] = ["—", precision.mean(), recall.mean(), f1.mean(), soporte.sum()]
df_metricas


In [ ]:
# Curvas ROC y AUC por clase (útil para justificar el umbral de decisión de la sección 10)
y_true_bin = label_binarize(y_true_idx, classes=range(NUM_CLASES))

plt.figure(figsize=(8, 7))
for i, clase in enumerate(CLASES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{clase} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("Tasa de falsos positivos")
plt.ylabel("Tasa de verdaderos positivos")
plt.title("Curvas ROC por clase (uno-contra-el-resto)")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()


## 10. Umbrales de decisión (confianza mínima del diagnóstico)

AgroDetect funciona como una **primera línea de apoyo al diagnóstico** (no reemplaza al
técnico agrónomo). Por eso es importante que, cuando el modelo **no esté seguro** de su
predicción, la app se lo indique al productor en lugar de dar un diagnóstico dudoso como si
fuera certero.

Para lograr esto, analizamos la **probabilidad softmax de la clase predicha** (confianza) y
definimos un **umbral de decisión**: si la confianza de la predicción es menor al umbral,
la recomendación del sistema es *"diagnóstico incierto — se recomienda consultar a un
técnico del IHCAFE"*, en vez de mostrar la clase con mayor probabilidad.


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(confianza_pred, bins=25, color="#4a7c3f", edgecolor="white")
plt.axvline(0.6, color="red", linestyle="--", label="Umbral propuesto = 0.60")
plt.title("Distribución de la confianza del modelo en sus predicciones (test)")
plt.xlabel("Confianza (probabilidad softmax de la clase predicha)")
plt.ylabel("Número de imágenes")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Análisis de cobertura vs. exactitud para distintos umbrales:
# - Cobertura: % de imágenes en las que el modelo SÍ da un diagnóstico (confianza >= umbral)
# - Exactitud entre las aceptadas: qué tan confiable es el modelo cuando SÍ se anima a decidir

umbrales = np.arange(0.30, 0.96, 0.05)
resultados_umbral = []

for u in umbrales:
    mask = confianza_pred >= u
    cobertura = mask.mean()
    if mask.sum() > 0:
        exactitud_aceptadas = accuracy_score(y_true_idx[mask], y_pred_idx[mask])
    else:
        exactitud_aceptadas = np.nan
    resultados_umbral.append({
        "umbral": round(u, 2),
        "cobertura": cobertura,
        "exactitud_en_aceptadas": exactitud_aceptadas,
        "n_diagnosticos_inciertos": int((~mask).sum())
    })

df_umbral = pd.DataFrame(resultados_umbral)
df_umbral


In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(df_umbral["umbral"], df_umbral["cobertura"], marker="o", color="#2e6b2e", label="Cobertura")
ax1.set_xlabel("Umbral de confianza")
ax1.set_ylabel("Cobertura (fracción de imágenes con diagnóstico)", color="#2e6b2e")
ax1.tick_params(axis="y", labelcolor="#2e6b2e")

ax2 = ax1.twinx()
ax2.plot(df_umbral["umbral"], df_umbral["exactitud_en_aceptadas"], marker="s", color="#a6520b", label="Exactitud en aceptadas")
ax2.set_ylabel("Exactitud entre diagnósticos aceptados", color="#a6520b")
ax2.tick_params(axis="y", labelcolor="#a6520b")

plt.title("Umbral de decisión: cobertura vs. exactitud")
fig.tight_layout()
plt.show()

UMBRAL_DECISION = 0.60
print(f"➡️  Umbral de decisión seleccionado para la app: {UMBRAL_DECISION}")
print("   (Ajustable según qué tan conservador se quiera que sea el sistema en campo)")


## 11. Prueba del modelo con imágenes

Probamos el modelo final con imágenes del conjunto de prueba, mostrando la **clase real**,
la **clase predicha**, la **confianza** y la **decisión final** aplicando el umbral definido
en la sección anterior — tal como se comportaría dentro de la app de Streamlit.


In [ ]:
def diagnosticar(imagen_array, modelo, umbral=UMBRAL_DECISION):
    '''Aplica el mismo flujo que usaría la app de Streamlit:
    preprocesa la imagen, predice y aplica el umbral de decisión.'''
    x = tf.image.resize(imagen_array, IMG_SIZE)
    x = preprocess_input(tf.expand_dims(x, 0))
    probs = modelo.predict(x, verbose=0)[0]
    idx = np.argmax(probs)
    confianza = probs[idx]
    if confianza < umbral:
        return "⚠️ Diagnóstico incierto — consultar a un técnico del IHCAFE", confianza, probs
    return CLASES[idx], confianza, probs

# Tomamos un lote de imágenes de prueba (sin preprocesar) para mostrarlas visualmente
muestras_test = []
for images, labels in test_ds_raw.take(1):
    for i in range(min(8, images.shape[0])):
        muestras_test.append((images[i].numpy(), CLASES[np.argmax(labels[i].numpy())]))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (img, clase_real) in zip(axes.flat, muestras_test):
    pred_clase, confianza, _ = diagnosticar(img, model)
    correcto = "✅" if pred_clase == clase_real else ("❓" if "incierto" in pred_clase else "❌")
    ax.imshow(img.astype("uint8"))
    ax.set_title(f"Real: {clase_real}\nPred: {pred_clase}\nConf: {confianza:.2f} {correcto}", fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Función lista para usarse con una imagen nueva subida por el usuario (ej. en Colab o Streamlit)
def diagnosticar_desde_archivo(ruta_imagen, modelo=model, umbral=UMBRAL_DECISION):
    img = Image.open(ruta_imagen).convert("RGB")
    img_array = np.array(img)
    pred_clase, confianza, probs = diagnosticar(img_array, modelo, umbral)

    plt.figure(figsize=(4, 4))
    plt.imshow(img_array)
    plt.title(f"{pred_clase}\nConfianza: {confianza:.2%}")
    plt.axis('off')
    plt.show()

    print("Probabilidades por clase:")
    for c, p in sorted(zip(CLASES, probs), key=lambda x: -x[1]):
        print(f"  {c:12s}: {p:.2%}")
    return pred_clase, confianza

# Ejemplo de uso (descomentar y ajustar la ruta al subir una imagen propia en Colab):
# from google.colab import files
# subida = files.upload()
# ruta = list(subida.keys())[0]
# diagnosticar_desde_archivo(ruta)


In [ ]:
print(mejor_nombre)

## 12. Exportación del modelo entrenado

Guardamos el modelo final (arquitectura + pesos) y la lista de clases, listos para
integrarse en la aplicación de **Streamlit** descrita en la sección 6 del anteproyecto.


In [ ]:
os.makedirs("/content/modelo_final", exist_ok=True)

# Guardamos el modelo en dos formatos:
#   - .keras : formato nativo recomendado por Keras 3 (mas completo y rapido de cargar)
#   - .h5    : formato HDF5 clasico, por compatibilidad (algunas versiones de Streamlit/
#              librerias externas o el codigo de despliegue pueden esperar este formato)
nombre_base = f"agrodetect_{mejor_nombre.lower()}"
MODEL_PATH_KERAS = f"/content/modelo_final/{nombre_base}.keras"
MODEL_PATH_H5 = f"/content/modelo_final/{nombre_base}.h5"

model.save(MODEL_PATH_KERAS)
model.save(MODEL_PATH_H5)

# Guardamos tambien el orden de clases, el umbral de decision, la arquitectura ganadora
# y la comparativa completa de las 3 arquitecturas evaluadas, para dejar constancia en
# el informe de por que se eligio esta arquitectura para el despliegue.
config_app = {
    "arquitectura_seleccionada": mejor_nombre,
    "clases": CLASES,
    "img_size": list(IMG_SIZE),
    "umbral_decision": UMBRAL_DECISION,
    "metricas_test": {
        "accuracy": float(test_acc),
        "precision": float(test_prec),
        "recall": float(test_rec),
    },
    "comparacion_arquitecturas": tabla_comparativa.to_dict(orient="records"),
    "fecha_entrenamiento": datetime.now().isoformat(),
}

with open("/content/modelo_final/config_app.json", "w", encoding="utf-8") as f:
    json.dump(config_app, f, ensure_ascii=False, indent=2)

print("✅ Modelo guardado en:", MODEL_PATH_KERAS)
print("✅ Modelo guardado en:", MODEL_PATH_H5)
print("✅ Configuración guardada en: /content/modelo_final/config_app.json")
print(json.dumps(config_app, ensure_ascii=False, indent=2))